# Women's Past Seasons Ratings

Calculate historical ratings of teams.

In [1]:
SEASON = 2025

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import optuna

pd.set_option('display.max_columns', 100)

def get_ratings(season):
    df = pd.read_parquet(fr'..\data\unprocessed\womens_sports_reference\full_season_sports_reference_{season}.parquet')
    
    # games at the beginning of the season should be about 0.8, games at the end will be 1.2
    df['Recency Bias'] = 0.996018**(df['Date'].max() - df['Date']).dt.days + 0.2

    X_offense = pd.get_dummies(
        df['Team']
    ).astype('int8')

    X_offense.columns += ' Offense'

    X_defense = -pd.get_dummies(
        df['Opponent']
    ).astype('int8')

    X_defense.columns += ' Defense'

    X = pd.concat([X_offense, X_defense], axis=1)

    X['Home Field Advantage'] = df['Location'].copy()

    def get_gkf_data(X, y, groups, cv=3):
        """
        Converts training data to list of folds
        """
        np.random.seed(22)
        gkf = GroupKFold(n_splits=cv)

        data = []
        for train_index, test_index in gkf.split(X, y, groups=groups):
            X_train = X[train_index, :]
            X_test = X[test_index, :]

            y_train = y[train_index]
            y_test = y[test_index]

            data.append([X_train, X_test, y_train, y_test])

        return data

    cv_data = get_gkf_data(X.to_numpy(), df['Team PPP'].to_numpy(), df['Date'].to_numpy())

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial, cv_data=cv_data):
        # model tuning
        alpha = trial.suggest_float('alpha', 0.1, 10, log=True)
        mod = Ridge(alpha=alpha)

        # cross validation
        y_actuals = []
        y_preds = []
        for X_train, X_test, y_train, y_test in cv_data:
            y_actuals.append(y_test)

            mod.fit(X_train, y_train)
            y_preds.append(mod.predict(X_test))

        return mean_squared_error(np.hstack(y_actuals), np.hstack(y_preds))

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=22))
    study.optimize(objective, n_trials=100, show_progress_bar=False)

    mod = Ridge(alpha=study.best_params['alpha'])

    mod.fit(X, df['Team PPP'], sample_weight=df['Recency Bias'])

    df_ppp = pd.DataFrame(
        {
            'Team': X.columns,
            'Rating': mod.coef_
        }
    ).sort_values(by=['Rating'], ascending=False, ignore_index=True)

    df_ppp_offense = df_ppp.loc[df_ppp['Team'].str.contains(' Offense$', regex=True), :].reset_index(drop=True)

    df_ppp_offense['Team'] = df_ppp_offense['Team'].str.replace(' Offense$', '', regex=True)

    df_ppp_offense['Rating'] += mod.intercept_

    df_ppp_defense = df_ppp.loc[df_ppp['Team'].str.contains(' Defense$', regex=True), :].reset_index(drop=True)

    df_ppp_defense['Team'] = df_ppp_defense['Team'].str.replace(' Defense$', '', regex=True)

    df_ppp_defense['Rating'] = mod.intercept_ - df_ppp_defense['Rating']

    df_ppp_all = pd.merge(
        df_ppp_offense.rename(columns={'Rating': 'Adjusted Offense'}),
        df_ppp_defense.rename(columns={'Rating': 'Adjusted Defense'}),
        how='inner',
        on=['Team']
    )

    df_ppp_all.insert(
        df_ppp_all.columns.get_loc('Adjusted Offense'), 
        'Efficiency Margin', 
        df_ppp_all['Adjusted Offense'] - df_ppp_all['Adjusted Defense'],
    )

    df_ppp_all.sort_values(['Efficiency Margin'], ascending=False, inplace=True, ignore_index=True)

    df_ppp_all.insert(0, 'Season', season)

    return df_ppp_all

In [ ]:
from tqdm.autonotebook import trange

df = pd.concat(
    [
        get_ratings(season)
        for season in trange(2008, SEASON)
    ],
    ignore_index=True,
)

df

  0%|          | 0/17 [00:00<?, ?it/s]

,Season,Team,Efficiency Margin,Adjusted Offense,Adjusted Defense
0,2008,Connecticut,0.553854,1.187382,0.633528
1,2008,Tennessee,0.444053,1.130261,0.686208
2,2008,Louisiana State,0.410704,1.061725,0.651022
3,2008,Stanford,0.393415,1.116417,0.723002
4,2008,North Carolina,0.365577,1.094338,0.728761
...,...,...,...,...,...
5874,2024,Wagner,-0.348871,0.692021,1.040891
5875,2024,Stonehill,-0.365483,0.699009,1.064492
5876,2024,Chicago State,-0.367008,0.727429,1.094438
5877,2024,South Carolina State,-0.374617,0.687214,1.061831


Some teams (like Ivy League in 2021) are missing, so add them in with NAs

In [6]:
df_full_teams = pd.DataFrame(
    [(season, team) for team in df['Team'].unique() for season in df['Season'].unique()],
    columns=['Season', 'Team']
)

df_full_teams

,Season,Team
0,2008,Connecticut
1,2009,Connecticut
2,2010,Connecticut
3,2011,Connecticut
4,2012,Connecticut
...,...,...
6166,2020,Le Moyne
6167,2021,Le Moyne
6168,2022,Le Moyne
6169,2023,Le Moyne


In [7]:
df = (
    pd.merge(
        df, 
        df_full_teams,
        how='right',
        on=['Season', 'Team']
    )
    .sort_values(
        ['Season', 'Team'], 
        ignore_index=True
    )
)

df

,Season,Team,Efficiency Margin,Adjusted Offense,Adjusted Defense
0,2008,Abilene Christian,NaN,NaN,NaN
1,2008,Air Force,-0.106985,0.834385,0.941370
2,2008,Akron,-0.128427,0.841616,0.970043
3,2008,Alabama,-0.063017,0.788375,0.851391
4,2008,Alabama A&M,-0.132963,0.802992,0.935955
...,...,...,...,...,...
6166,2024,Wright State,-0.047642,0.924843,0.972485
6167,2024,Wyoming,0.071691,0.969801,0.898109
6168,2024,Xavier,-0.205147,0.796599,1.001746
6169,2024,Yale,-0.099924,0.880534,0.980458


In [8]:
df['Past 4 Years Efficiency Margin'] = (
    df
    .groupby(['Team'])
    ['Efficiency Margin']
    .rolling(window=4, min_periods=2)  # at least 2 years of data to calculate
    .mean()
    .reset_index()
    .set_index('level_1')
)['Efficiency Margin']

df.rename(
    columns={
        'Efficiency Margin': 'Past Year Efficiency Margin'
    }, 
    inplace=True
)

df['Season'] += 1  # shift by a year so metrics are from past instead of the current rating

df = df.loc[df['Season'] >= 2012, :].reset_index(drop=True)

df

,Season,Team,Past Year Efficiency Margin,Adjusted Offense,Adjusted Defense,Past 4 Years Efficiency Margin
0,2012,Abilene Christian,NaN,NaN,NaN,NaN
1,2012,Air Force,-0.116981,0.871025,0.988006,-0.187368
2,2012,Akron,0.024465,0.900069,0.875604,-0.035480
3,2012,Alabama,0.052840,0.893265,0.840425,0.011622
4,2012,Alabama A&M,-0.141989,0.785509,0.927498,-0.161826
...,...,...,...,...,...,...
5077,2025,Wright State,-0.047642,0.924843,0.972485,-0.068898
5078,2025,Wyoming,0.071691,0.969801,0.898109,0.079227
5079,2025,Xavier,-0.205147,0.796599,1.001746,-0.105021
5080,2025,Yale,-0.099924,0.880534,0.980458,-0.042498


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5082 entries, 0 to 5081
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Season                          5082 non-null   int64  
 1   Team                            5082 non-null   object 
 2   Past Year Efficiency Margin     4895 non-null   float64
 3   Adjusted Offense                4895 non-null   float64
 4   Adjusted Defense                4895 non-null   float64
 5   Past 4 Years Efficiency Margin  4882 non-null   float64
dtypes: float64(4), int64(1), object(1)
memory usage: 238.3+ KB


Save

In [10]:
df.to_parquet(f'../data/preprocessed/womens_past_seasons/past_seasons_ratings.parquet')

'Done'

'Done'